# 599 - Is there a microphone on the Blackrock analog inputs?

**Why this matters more than it sounds.** There is *no speech-onset event anywhere in
this dataset*. Every timing result in stage 05 is locked to the **GO cue** - the moment
the stimulus ended - not to the moment the patient actually spoke. The gap between
those two **is** the response latency, and right now it is unmeasured.

Blackrock names its analog inputs `ainp1..ainp16`. The MicroEPI micro side is recorded
on Blackrock at 30 kHz. If a microphone was patched into one of those inputs, the
speech is in `raw_blackrock/*.ns6` - the only stream in this dataset fast enough to
carry audio, and the only route to a real speech onset.

> **NOTES:** *This notebook answers a yes/no question. If the answer is yes for even one
> patient, that patient can anchor a speech-onset analysis and calibrate what the GO-cue
> latency in [510](510_response_timing.ipynb) is actually measuring.*


## What was already ruled out

- The curated `*_export_Labs_*.mat` files contain **no ainp channels at all** on any of
  the six MicroEPI patients. They carry Micromed-style `X1..X9`, `MKR1..4`, `ECG1..2`
  plus a separate `photodiode` array. So the exports cannot answer this.
- `DBOut.mat` -> `DBEcog` is a database of **Micromed TRC files**, not channels.
  `DBMicro` is the Blackrock side and points at `raw_blackrock/*.ns6`.
- The `04_ersp_LM_RAWONLY` tree *does* hold ERSP cubes named `ainp1..ainp3` for
  PAT_6704 and PAT_6854 - evidence that an **earlier** export included them.

So the raw NSx has to be read directly. No Blackrock reader is installed;
`functions/lf_nsx.py` is a minimal one (header + channel labels + windowed seek).


In [ ]:
import sys, glob, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, 'functions')
import lf_nsx as N

RAW = Path(r'S:\HumanNeuronLab\DATARAW\MICROEPI')
PATS = ['G-01','G-02','G-03','G-04','G-05','G-06']

for pid in PATS:
    fs_ = sorted(glob.glob(str(RAW / f'MicroEPI-{pid}' / 'raw_blackrock' / '**' / '*.ns6'),
                           recursive=True))
    print(f'{pid}: {len(fs_)} .ns6 files')


## 1 - Do the ainp channels exist, and what is in them?

`scan_audio_channels.py` samples blocks spread across each file rather than one window
- a single slice lands in silence too easily and says nothing. Run it from a terminal
(it reads ~1 GB per file); this cell just loads what it wrote.

```bash
python scan_audio_channels.py --max-files 1 --n-blocks 6
python scan_audio_channels.py --patient G-04 --plot
```


In [ ]:
S = pd.read_csv('outputs/audio_scan/audio_channel_scan.csv')
V = json.loads(Path('outputs/audio_scan/verdict.json').read_text())
cols = ['patient','pat_name','file','channel','std','frac_100_4k','frac_4k_up',
        'env_2_8Hz','dyn_db']
display(S[[c for c in cols if c in S.columns]].round(3))
print(json.dumps(V['verdict'], indent=2))


### How to read those columns

| column | a live microphone | an unconnected input |
|---|---|---|
| `frac_100_4k` | **high** - speech energy lives here | low |
| `frac_4k_up` | low | **high** - amplifier hiss |
| `env_2_8Hz` | high - the syllable rate | undefined / flat |
| `dyn_db` | **15-40 dB** between silence and speech | **< 3 dB** |

`dyn_db` is the one that decides it. Speech in a task is intermittent; a noise floor is
not. A channel with a 0.5 dB dynamic range is not recording a room.


## 2 - Look at the whole file, not a summary

Envelope per channel across the entire NSx, plus a spectrogram of the strongest one.
Speech would appear as **repeated bursts** with harmonic structure below ~4 kHz.


In [ ]:
from IPython.display import Image, display as _d
for p in sorted(Path('outputs/audio_scan').glob('AUDIO_*.png')):
    print(p.name); _d(Image(str(p)))


## 3 - The sampling caveat, which is the whole game

> **NOTES:** *The scan above reads the **first** `.ns6` per patient. There are
> **2192-3511** files per patient, each 5 minutes. The language task is a ~25 minute
> block somewhere in that recording, and it is almost certainly **not** in file 001.*

So a "noise" verdict on file 001 does **not** mean the patient has no audio - it means
the microphone was not live during those five minutes. To settle it, the search has to
go to the files that actually cover the task.

Two routes to find them, in order of directness:


In [ ]:
# ROUTE A - DBMicro lists every Blackrock file with its sample count and events.
# The task block is the stretch whose events match the photodiode trial structure.
import types
sys.modules.setdefault('mne', types.ModuleType('mne'))
sys.path.insert(0, str(Path.cwd().parent / '01_FBM_Analysis'))
from functions import config as cfg
from scipy.io import loadmat

pid = 'G-04'
db = loadmat(Path(cfg.MICROEPI_MAT_PRESETS[pid]['data_dir']) / 'DBOut.mat',
             squeeze_me=True, struct_as_record=False)
M = db['DBMicro']
print(f'{pid}: DBMicro has {len(M)} Blackrock entries')
print('fields:', M[0]._fieldnames)
rows = [dict(ns6=str(getattr(e,'ns6filename','')), folder=str(getattr(e,'folder','')),
             n=int(getattr(e,'totSamples',0) or 0)) for e in M[:2000]]
D = pd.DataFrame(rows)
D['minutes'] = D['n'] / 30000 / 60
print(D['minutes'].describe().round(2).to_string())
display(D.head(8))


In [ ]:
# ROUTE B - brute scan: cheap header read + a short probe per file, keeping only
# files whose ainp dynamic range is speech-like. One header read is microseconds;
# the probe reads a few hundred kB, so a few thousand files is minutes, not hours.
def probe_file(p, t_probe=20.0):
    try:
        h = N.read_header(p)
    except Exception:
        return None
    ai = N.find_channels(h, 'ainp')
    if not ai:
        return None
    dur = h.n_samples / h.fs
    t0 = max(0.0, dur/2 - t_probe/2)
    X = N.read_window(h, ai, t0, t0 + t_probe, max_samples=int(t_probe*h.fs)+10)
    fs = N.effective_fs(h, t0, t0 + t_probe, max_samples=int(t_probe*h.fs)+10)
    best = None
    for k, i in enumerate(ai):
        x = X[k].astype(float)
        if x.size < fs or x.std() < 1e-9:
            continue
        w = max(1, int(fs*0.02))
        env = np.convolve(np.abs(x - x.mean()), np.ones(w)/w, 'same')
        dyn = 20*np.log10(max(np.percentile(env,95),1e-9)/max(np.percentile(env,10),1e-9))
        if best is None or dyn > best['dyn_db']:
            best = dict(file=Path(p).name, channel=h.chan_labels[i],
                        dyn_db=float(dyn), std=float(x.std()), dur_s=dur)
    return best

files = sorted(glob.glob(str(RAW / 'MicroEPI-G-04' / 'raw_blackrock' / '**' / '*.ns6'),
                         recursive=True))
print(f'{len(files)} files; probing every 25th as a first pass')
hits = [r for r in (probe_file(f) for f in files[::25]) if r]
H = pd.DataFrame(hits).sort_values('dyn_db', ascending=False)
display(H.head(15))
print('\nAny file with dyn_db > 12 dB is a real candidate — open it and listen/plot.')


## 4 - If a candidate turns up

Plot it against the trial structure. The photodiode gives stimulus onset and offset
(the GO cue); the audio envelope gives **speech onset**. The distance between the GO
cue and the first sustained rise in the audio envelope is the response latency this
whole folder is currently unable to measure.

> **NOTES:** *The Blackrock clock and the export/task clock are **not** the same. The
> exports are Micromed-derived and start at their own zero. Aligning the two needs the
> `ev` / `evParsed` fields in `DBMicro`, or a shared trigger present in both streams.
> Do not assume a common origin - that is exactly the class of error that produced the
> earlier trial-timing bugs.*


In [ ]:
# Template for a candidate, once one is found.
CAND_FILE = None      # e.g. .../20250402-170904-014.ns6
CAND_CHAN = 'ainp1'
if CAND_FILE:
    h = N.read_header(CAND_FILE)
    i = [j for j,l in enumerate(h.chan_labels) if l.lower()==CAND_CHAN][0]
    dur = h.n_samples/h.fs
    X = N.read_window(h, [i], 0, dur, max_samples=600_000)
    fs = N.effective_fs(h, 0, dur, max_samples=600_000)
    x = X[0].astype(float); x -= x.mean()
    w = max(1, int(fs*0.05))
    env = np.convolve(np.abs(x), np.ones(w)/w, 'same')
    fig, ax = plt.subplots(2,1, figsize=(13,6), sharex=True)
    ax[0].plot(np.arange(x.size)/fs, env, lw=.6)
    ax[0].set_ylabel(f'{CAND_CHAN} |env|')
    ax[1].specgram(x, NFFT=1024, Fs=fs, noverlap=512, cmap='magma')
    ax[1].set_ylim(0, 5000); ax[1].set_ylabel('Hz'); ax[1].set_xlabel('s (Blackrock clock)')
    plt.tight_layout(); plt.show()
else:
    print('no candidate set — run section 3 first')


## 5 - Where this leaves stage 05

> **NOTES:**
> - *If audio is found for even one patient, it calibrates everything: you can measure
>   the GO-to-speech delay directly for that patient and state what the GO-locked
>   latencies in 510 are offset by.*
> - *If no audio exists anywhere, that is a result too, and it should be written into
>   the caveats rather than left as an open question - it means GO-locked is the ceiling
>   for this dataset and no reviewer request can change it.*
> - *Only the six MicroEPI patients have Blackrock at all. The EL and PAT cohorts are
>   Micromed-only, so even a positive result here covers 6 of 29 patients.*
